#Spark Learning

##By Knowing the process of read and write , we become a Data ingestion developer

connecting various sources (files or filesystem , db , dwh , api ,.. ) and loading data into storage env(data lake)

- csv
- json
- xml
- parquet
- ORC
- sql etc....


In [0]:
#To know Spark version

spark.version


## Few Facts about Unity Catalog

Dataobjects  is managed  in catalog with three namespace 


Catalog -> per domain or environemnt 

schema -> database 

tables , views , functions , volume 

Volume -  Non tabular data (files) , goverened access 

Catalog >> Schema  >> 
                    Table
                    View
                    functions
                    volume 

Volumes are used for managing Non tabular data (files)

In [0]:
%sql
create catalog if not exists izwd37dev;

create schema if not exists izwd37dev.wd37db;

--create volume if not exists izwd37dev.wd37db.rawdata;

create volume if not exists izwd37dev.wd37db.rawdatta;


## DBFS 

### Databricks File system 

distribuited virtual file system , linux posix format  runninng on top of your cloud storages 


/Volumes/catalog/schema/volume-name/path/to/file



dbfs:/ - uri   -> uniform resource identifier  (dbricks file system )

hdfs:/    -> hadoop distruibuited file system 

file:/     -> local file 

s3a:/   -> aws s3

gcs:/  -> google storage


adls:/   -> azure datalake 

In [0]:
%fs ls "dbfs:/Volumes/izwd37dev/wd37db/rawdata/"

In [0]:
%sql
drop volume izwd37dev.wd37db.rawdata

### Volumes operations using fs commands
-  List out files in Volumes ( Unity Catalog)

In [0]:
%fs ls "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/"

In [0]:
%fs ls "/Volumes/izwd37dev/wd37db/rawdatta/cust_sample.txt"

In [0]:
print(dbutils.fs.ls("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/"))

In [0]:
%fs mkdirs "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv"


In [0]:
%fs mkdirs "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json"
--dbutils.fs.mkdirs("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json")

what is dbfs:/ -> uri 
        hdfs:/ 
        file:/
        s3:/
        gcs:/

In [0]:
#read data from spark 
#spark SQL 

Spark Session

from pyspark.sql import SparkSession


spark = SparkSession.builder().getOrCreate()


In [0]:
print(spark)

create Dataframe from storage (files / dir ) To read the delimited data from any storage (dbfs , hdfs , lfs , cloud storages )

spark.read.csv opition -> dataframe

-- csv is the built in source

Loaded the custs file into rawdatta volume 

In [0]:
%fs ls "dbfs:/Volumes/izwd37dev/wd37db/rawdatta"

In [0]:
custdata = spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs") #file path
print(type(custdata)) #dataframe
custdata.show() #show is action, similar to collect 
# show action , display default 20 records 

In [0]:
#Describe table
custdata.printSchema()

In [0]:
#view few more records
custdata.show(10,False)
#custdata.show(200)                                   

In [0]:
#Supported in Databricks to show result in formatted wy to filter, ascend, descend data 
display(custdata)

In [0]:
#custdata is a dataframe and Schema provides schema format in spark Structtype and Structfiled type 
print(custdata.schema)

In [0]:
#Changing default column name _c0,_c1 etc to actual custid, fname etc
custdata= spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs").toDF("custid","fname","lname","age","profession")

custdata.show(5,False)  #Display only 5 records
custdata.printSchema()

### suppose we recivied a file with header
### column names we need to pick from the header

In [0]:
#Without header option
custdata = spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header")

custdata.show(5)

custdata.printSchema()

In [0]:
%python
# how to get the number of records in the dataframe
# select count(1) from table 
# count is an action , its return integer result , trigger execution , job is created
custdata.count()

overriding the defaults with option

enable the header

In [0]:
#With header option
custdata = spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header",header=True)

custdata.show(5)

custdata.printSchema()

print(f"Record count in custs is {custdata.count()}")

In [0]:
# data type for all columns treated default as string
# based on the data we have generate the schema with proper data type 
# performance if we read large data inferSchema is not a good option 
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header",header=True,inferSchema=True)

custdata.show(5)

custdata.printSchema()


## Different delimited (|)

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/emp1.csv

In [0]:
emp_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/emp1.csv",header=True,inferSchema=True,sep="|")
emp_df.show(5)
emp_df.printSchema()

spark.read.csv()

-> required input **path**

-> path could be a file or dir , list of dir ...

In [0]:
# options , way to create dataframe using csv with different options
custdata= spark.read.csv(path="/Volumes/izwd37dev/wd37db/rawdatta/csv/") # reading directory
custdata.show(5)
custdata.printSchema()

In [0]:
#/Volumes/izwd37dev/wd37db/rawdatta/csv/cust2.csv
#/Volumes/izwd37dev/wd37db/rawdatta/csv/custs

#Reading and creating datatframe from list of paths - directory and files lists

custdata=spark.read.csv(path=["/Volumes/izwd37dev/wd37db/rawdatta/csv/custs","dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/cust2.csv"]) # reading list of path/files
custdata.show(5,True)
custdata.printSchema()
custdata.count()


In [0]:
%fs head "/Volumes/izwd37dev/wd37db/rawdatta/salesdata/chennai/sales_chennai.csv"

In [0]:
#Reading files with pattern matching using '*'
sales_df = spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/salesdata/chennai/sales*", header=True,inferSchema=True)
sales_df.show(5)
print(sales_df.count())
sales_df.printSchema()

In [0]:
# recursiveFileLookup - read all sub dir
# pathGlobFilter - apply the filter / pattern globally on all dir /sub dir

sales_df= spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/salesdata", header=True,recursiveFileLookup=True,pathGlobFilter="sales*")

sales_df.show(4)
sales_df.printSchema()
print(sales_df.count())

In [0]:
%fs ls /Volumes/izwd37dev/wd37db/rawdatta/salesdata/chennai

In [0]:
sales_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/salesdata",header=True,recursiveFileLookup=True,pathGlobFilter="sales*")

sales_df.count()

In [0]:
%fs head  /Volumes/izwd37dev/wd37db/rawdatta/emp1.csv
%fs head  /Volumes/izwd37dev/wd37db/rawdatta/custs

In [0]:
# inferSchema -> generating schema by reading the entire data 
# to avoid reading the data for genrating schema , - performance issue 
# when we are going with inferschema its more dynamic  - dq issue 

emp_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/emp1.csv",header=True,inferSchema=True,sep="|")
emp_df.show(5)
emp_df.printSchema()

### we can create our own schema and apply while reading data

In [0]:
# simple way to define schema 
'''
create table cust(custid int,fname string,lname string,age int , prof string)
'''
# creating the schema using ddl string / string
cust_schema="custid int,fname string,lname string,age int , prof string"
cust_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/custs",schema=cust_schema)
cust_df.show(5)
cust_df.count()
cust_df.printSchema()

In [0]:
schema_string="eid integer,name string,age int"
emp_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/emp1.csv",header=True,sep="|",schema=schema_string)
emp_df.show(5)
emp_df.printSchema()

print(emp_df.schema)

In [0]:
# programmatically define the schema - using StructType and StructField

from pyspark.sql.types import StructType,StructField,IntegerType,StringType

'''
int -> integrType
str -> StringType

StructType - structure (row) / record 
StructField - column -> [name , type ]
'''

schema_string = StructType(
    [ StructField("eid",IntegerType(),True),
      StructField("name",StringType(),True),
       StructField("age",IntegerType(),True)
    ] )

emp_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/emp1.csv",header=True,sep="|",schema=schema_string)
emp_df.show(5)
emp_df.printSchema()



## built in Source

- CSV
- JSON
- XML 
- parquet 
- delta
- ORC
- Excel
- jdbc
- table

## Read data from JSON

Javascript object notation 

semi structure data 

key value data 

along with data , it will keep the field name as well

key - colum , value -data 

column ordering not important , number of fields also dynamic 

In [0]:
%fs head "/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json"

In [0]:
emp_schema = "eid int,name string, age int"
emp_df = spark.read.json("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json",schema=schema_string)
emp_df.show(5)
emp_df.printSchema()
print(f"Count of df is {emp_df.count()}")
print(emp_df.schema)

#note : we defined above schema after checking json default schema behaviour ( as below)
#json provides below schema with Long as defaul for integer without scheme defined. 
#StructType([StructField('age', LongType(), True), StructField('id', LongType(), True), StructField('name', StringType(), True)])


In [0]:
# connect external source 
# genric way to read data using spark

# spark.read.option("k","v").format("source").load()

# csv 

# df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/custs")
#"dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header"

cust_df=spark.read.option("header","True").option("inferSchema","True").format("csv").load("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header")

cust_df.show(5)
cust_df.printSchema()
cust_df.count()



In [0]:
#%fs head /Volumes/izwd37dev/wd37db/rawdatta/json/emp.json

emp_schema = "id int, name string, age int"
emp_df= spark.read.schema(emp_schema).format("json").load("/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json")
emp_df.show()
emp_df.printSchema()

In [0]:
# create dataframe from delimited file , comes from any storage 
# local 
# hdfs 
# cloud storages - s3 , gcs , adls
# databricks file system - dbfs

#/Volumes/izwd37dev/wd37db/rawdatta/custs_header

cust_db = spark.read.option("inferSchema","True").csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header",header=True)
cust_db.show(4)
cust_db.printSchema()

In [0]:
%fs head "/Volumes/izwd37dev/wd37db/rawdatta/custjson/custdata.json"

In [0]:
cust_json_db= spark.read.json("/Volumes/izwd37dev/wd37db/rawdatta/custjson/custdata.json")
cust_json_db.show()
cust_json_db.printSchema()
cust_json_db.count()

In [0]:
cust_xml_df= spark.read.xml("/Volumes/izwd37dev/wd37db/rawdatta/custxml/custdata.xml",rowTag='customer')
cust_xml_df.show(5)
cust_xml_df.printSchema()


In [0]:
#/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv
cust_schema="sid int,sname string,dept string,yearofbirth int"
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv", schema=cust_schema,header=True)
cust_df.show()
cust_df.printSchema()
cust_df.count()

# bad record (not matching the schema )
# allow - default 
# fail 
# ignore 

option -> mode 
1. permissive -> permitting , RCA later
2. dropmalformed - blindly drop that record proceed 
3. failfast - immediately fail the whole job

In [0]:
stud_schema="sid int,sname string,dept string,yearofbirth int"
stud_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv",header=True,schema=stud_schema,mode="PERMISSIVE")
stud_df.show()
stud_df.printSchema()

In [0]:
stud_schema="sid int,sname string,dept string,yearofbirth int"
stud_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv",header=True,schema=stud_schema,mode="dropMalformed")
stud_df.show()
# print(stud_df.count())- 5 record will show actual we have only 3
stud_df.printSchema()

In [0]:
stud_schema="sid int,sname string,dept string,yearofbirth int"
stud_df = spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv",header=True,schema=stud_schema, mode="failFast")
stud_df.show()
stud_df.printSchema()

In [0]:
stud_schema="sid int,sname string,dept string,yearofbirth int,error_rec string"
#stud_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv",header=True,schema=stud_schema,mode="PERMISSIVE",columnNameOfCorruptRecord="error_rec")
stud_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv",header=True,schema=stud_schema,mode="Permissive",columnNameOfCorruptRecord="error_rec")
stud_df.show()